# Simplified GPT-Style Decoder in PyTorch

This notebook builds a small decoder-only language model that follows the main ideas behind GPT-style models without trying to exactly reproduce GPT-2.

What this notebook covers:
- masked self-attention
- residual connections and layer normalization
- token embeddings plus position information
- next-token training on a character-level dataset
- text generation from a partial prompt

The goal is to keep the model small enough to study while still showing the full training and generation workflow.

In [55]:
%pip install torch numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [56]:
# import packages
from typing import Optional
from urllib.request import urlopen

import torch
print(torch.__version__)
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

print(torch.__version__)
print(np.__version__)

2.11.0+cu130
2.11.0+cu130
2.2.6


### Notebook Roadmap

We will move through the notebook in four parts:

1. Build a single transformer block with causal self-attention.
2. Stack those blocks into a decoder-only language model.
3. Prepare a simple character-level Tiny Shakespeare dataset.
4. Train the model and inspect generation during training.

In [57]:
# A single decoder block: masked self-attention followed by a feed-forward network.
class TransformerBlock(nn.Module):
    def __init__(self, 
                 hid_dim: int, 
                 head_num: int,
                 linearmap_dim: int,
                 dropout: float = 0.1
                 ):
        super().__init__()
        self.hid_dim = hid_dim
        self.head_num = head_num
        
        if hid_dim % head_num != 0:
            raise ValueError(f"The hidden dimension {self.hid_dim} can not be divided by {self.head_num}")
        else:
            self.head_dim = hid_dim // head_num

        # Project the hidden states into query, key, and value spaces.
        self.q_map = nn.Linear(hid_dim, hid_dim)
        self.k_map = nn.Linear(hid_dim, hid_dim)
        self.v_map = nn.Linear(hid_dim, hid_dim)
        
        # Project the concatenated attention heads back to the model dimension.
        self.o_map = nn.Linear(hid_dim, hid_dim)
        
        self.layernorm1 = nn.LayerNorm(hid_dim)
        self.layernorm2 = nn.LayerNorm(hid_dim)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.mlp_dropout = nn.Dropout(dropout)
        
        self.f_map = nn.Sequential(
            nn.Linear(hid_dim, linearmap_dim),
            nn.GELU(),
            nn.Linear(linearmap_dim, hid_dim)
        )
        
    def forward(self, 
                q: torch.Tensor,  # (B, l_q, hid_dim)
                k: torch.Tensor,  # (B, l_s, hid_dim)
                v: torch.Tensor,  # (B, l_s, hid_dim)
                mask: Optional[torch.Tensor] = None  # (1 or B, 1 or head_num, l_q, l_s)
                ) -> torch.Tensor: 
        
        B, l_q, _ = q.shape
        _, l_s, _ = k.shape
        
        q_mapped = self.q_map(q)
        k = self.k_map(k)
        v = self.v_map(v)
        
        # Reshape from (B, L, hid_dim) to (B, head_num, L, head_dim).
        q_mapped = q_mapped.view(B, l_q, self.head_num, self.head_dim).transpose(1, 2)
        k = k.view(B, l_s, self.head_num, self.head_dim).transpose(1, 2)
        v = v.view(B, l_s, self.head_num, self.head_dim).transpose(1, 2)
         
        score = torch.matmul(q_mapped, k.transpose(-2, -1)) / self.head_dim ** 0.5
        
        # The causal mask prevents each position from attending to future tokens.
        if mask is not None:
            score = score.masked_fill(~mask, float("-inf"))
        
        score = torch.softmax(score, dim=-1)
        score = self.attn_dropout(score)
        
        score = torch.matmul(score, v)
        output = score.transpose(1, 2).contiguous().view(B, l_q, self.hid_dim)
        
        # First residual branch: attention output plus the original input.
        output = self.o_map(output)
        output = self.resid_dropout(output)
        output = self.layernorm1(q + output)
        
        # Second residual branch: feed-forward network on top of the attention result.
        mlp_output = self.f_map(output)
        mlp_output = self.mlp_dropout(mlp_output)
        output = self.layernorm2(output + mlp_output)

        return output

In [58]:
# Stack several transformer blocks into a decoder-only language model.

class Decoder(nn.Module):
    def __init__(self,
                 layer_num: int,
                 
                 # Model size
                 vocab_size: int,
                 hid_dim: int,
                 head_num: int,
                 linearmap_dim: int,
                 dropout: float = 0.1,
                 max_length: int = 1024,
                 ):
        super().__init__()
        
        self.layer_num = layer_num
        self.vocab_size = vocab_size
        self.hid_dim = hid_dim
        self.head_num = head_num
        self.linearmap_dim = linearmap_dim
        self.max_length = max_length
        
        # Each token id is mapped into the model hidden space.
        self.token_embedding = nn.Embedding(vocab_size, hid_dim)
        self.embed_dropout = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList(
            [TransformerBlock(hid_dim, head_num, linearmap_dim, dropout) for _ in range(layer_num)]
        )
        
        # Final projection produces one logit per vocabulary item.
        self.final_map = nn.Linear(hid_dim, vocab_size)
        
    def _position_encode(self, 
                         input: torch.Tensor
                         ) -> torch.Tensor: 
        B, l_q, _ = input.shape
        
        if l_q > self.max_length:
            raise ValueError(
                f"Sequence length {l_q} exceeds max_length {self.max_length}"
            )
        
        # Use sinusoidal position encoding to keep the example lightweight.
        positions = torch.arange(l_q, device=input.device, dtype=input.dtype).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, self.hid_dim, 2, device=input.device, dtype=input.dtype)
            * (-torch.log(torch.tensor(10000.0, device=input.device, dtype=input.dtype)) / self.hid_dim)
        )
        
        position_encoding = torch.zeros(l_q, self.hid_dim, device=input.device, dtype=input.dtype)
        position_encoding[:, 0::2] = torch.sin(positions * div_term)
        position_encoding[:, 1::2] = torch.cos(
            positions * div_term[: position_encoding[:, 1::2].shape[1]]
        )
        
        return input + position_encoding.unsqueeze(0)
        
    def _causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        # A lower-triangular mask lets each token see only itself and earlier tokens.
        return torch.tril(torch.ones((1, 1, seq_len, seq_len), device=device, dtype=torch.bool))
        
    def forward(self, 
                input_ids: torch.Tensor,
                mask: Optional[torch.Tensor] = None
                ) -> torch.Tensor:
        # input_ids: (B, L)
        # output logits: (B, L, vocab_size)
        output = self.token_embedding(input_ids)
        output = self._position_encode(output)
        output = self.embed_dropout(output)
        
        if mask is None:
            mask = self._causal_mask(input_ids.shape[1], input_ids.device)
        
        for block in self.blocks:
            output = block(output, output, output, mask)
        
        output = self.final_map(output)
        return output

### Tiny Shakespeare Dataset

For a compact GPT-style demo, Tiny Shakespeare works well because we can use character-level tokenization and avoid an external tokenizer library.

Key idea:
- each unique character becomes one token
- `vocab_size` is the number of unique characters in the text
- the model input has shape `(batch_size, sequence_length)`
- the model output has shape `(batch_size, sequence_length, vocab_size)`

This means the final linear layer must map from `hid_dim` to `vocab_size`, because each position predicts the next character.

In [59]:
# Character-level dataset built from Tiny Shakespeare.
class SimpleDataSet(Dataset):
    def __init__(self,
                 sequence_length: int = 800
                 ):
        # Download the raw text once and keep only full fixed-length chunks.
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        text = urlopen(url).read().decode("utf-8")

        self.sequence_length = sequence_length
        self.samples_num = len(text) // self.sequence_length
        self.text = text[: self.samples_num * self.sequence_length]

        self.chars = sorted(set(text))
        self.vocab_size = len(self.chars)

        self.stoi = {char: idx for idx, char in enumerate(self.chars)}
        self.itos = {idx: char for char, idx in self.stoi.items()}

    def _tokenizer(self,
                  input_text: str,
                  ) -> torch.Tensor:
        # Convert a string into integer token ids.
        tokens = []
        for _char in input_text:
            tokens.append(self.stoi[_char])
        return torch.tensor(tokens, dtype=torch.long)

    def _detokenizer(self,
                    input_token: torch.Tensor
                    ) -> str:
        # Convert integer token ids back into characters.
        return "".join(self.itos[int(token)] for token in input_token)

    def __len__(self):
        return self.samples_num

    def __getitem__(self, index):
        # Return one fixed-length chunk of text and its token ids.
        _text = self.text[index * self.sequence_length:(index + 1) * self.sequence_length]
        _token_ids = self._tokenizer(_text)
        return _token_ids, _text

In [60]:
# Build the dataset and a small decoder model for the demo.
batch_size = 32
dataset = SimpleDataSet(sequence_length=800)
print(f"Dataset size: {len(dataset)}")
dataloader = DataLoader(dataset, batch_size=batch_size)

# Keep the model small enough that the notebook remains easy to inspect.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
layer_num = 4
sequence_length = 800
hid_dim = 32
head_num = 4
linearmap_dim = 1280
dropout = 0.1
vocab_size = dataset.vocab_size

print(f"Vocabulary size: {vocab_size}")

decoder = Decoder(
    layer_num=layer_num,
    vocab_size=vocab_size,
    hid_dim=hid_dim,
    head_num=head_num,
    linearmap_dim=linearmap_dim,
    dropout=dropout,
    max_length=sequence_length,
    ).to(device)

parameters = sum(parameter.numel() for parameter in decoder.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in decoder.parameters() if parameter.requires_grad
)
print(f"Amount of parameters: {parameters}")
print(f"Amount of trainable parameters: {trainable_parameters}")

Dataset size: 1394
Vocabulary size: 65
Amount of parameters: 354561
Amount of trainable parameters: 354561


In [61]:
# Helper functions for qualitative generation checks during training.
def generate_text(model, prompt_ids, max_new_tokens, temperature=2.0, greedy=False):
    device = next(model.parameters()).device
    generated = prompt_ids.to(device)
    was_training = model.training

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # If the sequence grows too long, keep only the most recent context window.
            context = generated[:, -model.max_length:]
            logits = model(context)
            next_token_logits = logits[:, -1, :] / temperature

            if greedy:
                next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            else:
                probs = torch.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

            generated = torch.cat([generated, next_token], dim=1)

    if was_training:
        model.train()

    return generated.cpu()

def preview_half_sequence_generation(
    model,
    token_ids,
    detokenizer,
    preview_total_tokens=160,
    temperature=1.0,
    greedy=False,
    ):
    # Use a shorter excerpt inside training so the notebook stays responsive.
    preview_ids = token_ids[:, : min(token_ids.shape[1], preview_total_tokens)]
    prompt_len = preview_ids.shape[1] // 2
    prompt_ids = preview_ids[:, :prompt_len]
    target_ids = preview_ids[:, prompt_len:]

    generated_ids = generate_text(
        model,
        prompt_ids,
        max_new_tokens=target_ids.shape[1],
        temperature=temperature,
        greedy=greedy,
    )

    prompt_text = detokenizer(prompt_ids[0])
    target_text = detokenizer(target_ids[0])
    generated_text = detokenizer(generated_ids[0, prompt_len:])

    print("=========================Prompt (first half)=========================")
    print(prompt_text)
    print("\n =========================Ground-truth continuation=========================")
    print(target_text)
    print("\n=========================Generated continuation=========================")
    print(generated_text)

### Training and Generation Preview

The training objective is standard next-token prediction: given tokens up to position `t`, predict the token at position `t + 1`.

To make the notebook more informative, the training loop also prints a short generation preview from half of a small excerpt. The preview is intentionally shorter than the full 800-character training sequence so the notebook remains responsive on CPU.

In [62]:
# Training configuration for a short, readable demo.
epoch = 1000
lr = 1e-3
preview_every = 10

optimizer = torch.optim.AdamW(decoder.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

for _epoch in range(epoch):
    for index, (input_ids, input_text) in enumerate(dataloader):
        decoder.train()
        optimizer.zero_grad()

        # Teacher forcing: use tokens 0..L-2 to predict tokens 1..L-1.
        input_ids = input_ids.to(device)
        logits = decoder(input_ids[:, :-1])
        targets = input_ids[:, 1:]
        loss = criterion(logits.transpose(1, 2), targets)

        loss.backward()
        optimizer.step()

    if _epoch % preview_every == 0:
        print(f"epoch {_epoch}, loss {loss.item():.4f}")
        preview_half_sequence_generation(
            decoder,
            input_ids[:1],
            dataset._detokenizer,
            greedy=False,
        )
        print("-" * 80)

epoch 0, loss 3.0554
=========================Prompt (first half)=========================
mps,
Fill all thy bones with aches, make thee roar
That beasts shall tremble at 

 =========================Ground-truth continuation=========================
thy din.

CALIBAN:
No, pray thee.
I must obey: his art is of such power,
It woul

=========================Generated continuation=========================
jtns wgs, q:G' s I und '.u tshy u F, G ntYbloZI ibiBiWemmeFnm bwJn, rizd  Zq- ho
--------------------------------------------------------------------------------
epoch 10, loss 2.5325
=========================Prompt (first half)=========================
mps,
Fill all thy bones with aches, make thee roar
That beasts shall tremble at 

 =========================Ground-truth continuation=========================
thy din.

CALIBAN:
No, pray thee.
I must obey: his art is of such power,
It woul

=========================Generated continuation=========================
fing
Ff whalonouspay t hash

KeyboardInterrupt: 